# บทที่ 07 - แผนผัง Design Pattern

โน้ตบุ๊กนี้สาธิต **Planning Design Pattern** สำหรับเอเย่นต์ AI โดยใช้ Microsoft Agent Framework
คุณจะได้เรียนรู้วิธีการแบ่งคำขอการเดินทางที่ซับซ้อนออกเป็นงานย่อยที่มีโครงสร้าง กำหนดให้กับเอเย่นต์เฉพาะทาง
และดำเนินแผนที่ได้จากขั้นตอนทั้งหมด — โดยใช้ผลลัพธ์ที่มีโครงสร้างซึ่งขับเคลื่อนด้วยโมเดล Pydantic


## การตั้งค่า


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## การแบ่งงาน

การแบ่งงานเป็นหัวใจสำคัญของรูปแบบการออกแบบแผนแทนที่จะให้ตัวแทนเพียงคนเดียว
จัดการคำขอที่ซับซ้อนตั้งแต่ต้นจนจบ เราจะแบ่งปัญหาออกเป็น **งานย่อย** ที่มีขอบเขตชัดเจนขนาดเล็กลง
งานย่อยแต่ละงานถูกมอบหมายให้ตัวแทนผู้เชี่ยวชาญเฉพาะด้าน (เช่น เที่ยวบิน โรงแรม กิจกรรม) พร้อมด้วย
ลำดับความสำคัญและการขึ้นต่อกันที่ชัดเจน

แนวทางนี้ให้ประโยชน์หลายประการ:
- **ความชัดเจน**: งานย่อยแต่ละงานมีความรับผิดชอบเพียงอย่างเดียว
- **ความขนาน**: งานย่อยที่ไม่ขึ้นต่อกันสามารถทำงานพร้อมกันได้
- **ความน่าเชื่อถือ**: ความล้มเหลวจำกัดอยู่เฉพาะงานย่อยแต่ละงาน
- **การติดตามงบประมาณ**: ต้นทุนถูกประมาณต่อแต่ละงานย่อยและรวมยอด


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## การสร้างเอเจนต์วางแผนด้วยผลลัพธ์ที่มีโครงสร้าง

เอเจนต์วางแผนทำหน้าที่เป็น **ผู้ประสานงานด้านหน้าสำนักงาน** โดยเมื่อได้รับคำขอการเดินทางในระดับสูง
จะสร้าง `TravelPlan` ที่มีโครงสร้างออกมา — โดยแยกคำขอเป็นงานย่อย กำหนดลำดับความสำคัญ
และระบุความสัมพันธ์ในการทำงาน เพื่อให้คอนเซียร์จหรือชั้นการดำเนินงานสามารถทำงานนั้นได้


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## การดำเนินแผนงานด้วยเครื่องมือผู้เชี่ยวชาญ

เมื่อตัวแทนหน้าเคาน์เตอร์ได้สร้างแผนงานที่มีโครงสร้างแล้ว **ตัวแทนคอนเซียจ** จะเป็นผู้ดำเนินการแผนงานนั้น
เครื่องมือผู้เชี่ยวชาญแต่ละอย่างจะจัดการกับหมวดหมู่ของงานย่อย (เที่ยวบิน โรงแรม กิจกรรม) ตัวแทนคอนเซียจ
จะทำซ้ำผ่านงานย่อยของแผนตามลำดับความขึ้นต่อกันและส่งแต่ละงานไปยังเครื่องมือที่เหมาะสม


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## สรุป

ในบทเรียนนี้คุณได้เรียนรู้ **แพทเทิร์นการออกแบบการวางแผน** สำหรับตัวแทน AI:

1. **การแบ่งงานเป็นส่วนย่อย** — ตัวแทนวางแผนแผนกต้อนรับจะแบ่งคำขอเดินทางที่ซับซ้อนออกเป็น
   งานย่อยที่มีโครงสร้างโดยใช้โมเดล Pydantic มอบหมายแต่ละงานให้ตัวแทนผู้เชี่ยวชาญพร้อมลำดับความสำคัญ
   และความขึ้นต่อกัน
2. **ผลลัพธ์ที่มีโครงสร้าง** — โดยการส่งผ่าน `response_format` ตัวแทนจะส่งคืนอ็อบเจ็กต์
   `TravelPlan` ที่ผ่านการตรวจสอบแทนข้อความเสรี ซึ่งทำให้กระบวนการต่อไปน่าเชื่อถือ
3. **การดำเนินการตามแผน** — ตัวแทนผู้ดูแลห้องพักจะทำงานตามลำดับงานย่อยโดยใช้เครื่องมือผู้เชี่ยวชาญ
   (`book_flight`, `reserve_hotel`, `book_activity`) เพื่อดำเนินการตามแผนและรายงานผลลัพธ์

แพทเทิร์นนี้แยก *สิ่งที่ต้องทำ* (การวางแผน) ออกจาก *วิธีการทำ* (การดำเนินการ) ทำให้ตัวแทน
มีความยืดหยุ่น ทดสอบง่าย และขยายได้ง่ายขึ้น


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
